# Project Setup

## Overview
This notebook is part of a reproducible GRPO-based fine-tuning pipeline for cybersecurity policy generation.

## Purpose of this setup cell
The first code cell in this notebook:
- loads environment variables from `.env`
- identifies the project root directory automatically
- defines standard folder paths used across the project
- creates required folders if they do not already exist

## Why this matters
This makes the notebook portable across macOS, Windows, and Linux without requiring users to manually edit file paths.

## Expected project folders
- `data/corpus/` → source PDF corpus
- `data/processed/` → intermediate processed data
- `data/sample/` → optional small example data
- `outputs/completions/` → generated completions
- `outputs/rankings/` → ranked outputs
- `outputs/models/` → trained model checkpoints
- `outputs/evaluations/` → evaluation results

### Initial Settings

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Define PROJECT_ROOT automatically
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Define folders
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "corpus"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
COMPLETIONS_DIR = OUTPUT_DIR / "completions"
RANKINGS_DIR = OUTPUT_DIR / "rankings"
MODELS_DIR = OUTPUT_DIR / "models"
EVAL_DIR = OUTPUT_DIR / "evaluations"

# Create folders if needed
for folder in [
    DATA_DIR, CORPUS_DIR, PROCESSED_DIR,
    OUTPUT_DIR, COMPLETIONS_DIR, RANKINGS_DIR, MODELS_DIR, EVAL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 06 Step 6 — Run the Multi-Agent Cybersecurity Workflow

## Purpose
This notebook orchestrates the multi-agent cybersecurity workflow using CrewAI and the configured model stack.

## Why this step is important
This step demonstrates how the trained or selected model can be applied in a practical multi-agent policy generation setting, where different agents contribute specialized roles.

## Inputs
- OpenAI API key from `.env`
- Ranked or model-generated policy artifacts
- Optional local LLaMA-compatible endpoint, if required by the notebook setup

## Outputs
- Multi-agent policy generation results
- Agent interaction outputs and logs

## Main tasks
1. Load environment variables
2. Initialize the model access layer
3. Configure the multi-agent system
4. Run coordinated agent tasks
5. Save the final generated outputs

## Success criteria
This step is complete when the agents run successfully and produce a final coordinated output.

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importing CrewAI libraries
from crewai import Agent, Task, Crew



### Setting the LLMs to be used by the Agents

In [3]:
# Environment check
import os

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("API key is set correctly.")
else:
    print("API key is not set.")

API key is set correctly.


In [4]:
# Llama API setup
import requests

def call_llama(prompt):
    """Call the local Llama model for processing."""
    url = "http://localhost:1234/v1/models"
    headers = {"Content-Type": "application/json"}
    payload = {
        "prompt": prompt,
        "temperature": 0.7,
        "max_tokens": 150,
    }
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    return response.json().get("text", "")

### Defining the Agents

In [5]:
# Defining Agents

policy_writer_agent = Agent(
    role="Policy Writer",
    goal="Draft and improve cybersecurity policies.",
    backstory=(
        "As a seasoned cybersecurity professional working for {organization}, "
        "your mission is to draft detailed cybersecurity policies "
        "based on collected datasets and align them with the latest cyber security standards."
    ),
    verbose=True,
    # model= 'gpt-4-turbo'
    model=call_llama 
)

policy_reviewer_agent = Agent(
    role="Policy Reviewer",
    goal="Review and refine cybersecurity policies.",
    backstory=(
        "An experienced cybersecurity reviewer, skilled in providing feedback and "
        "ensuring adherence to regulations and standards."
    ),
    verbose=True,
    # model= 'gpt-4-turbo'
    model=call_llama 
)

compliance_policy_mapper_agent = Agent(
    role="Compliance-Policy Mapper",
    goal="Map policies to compliance standards and create action plans for gaps.",
    backstory=(
        "A compliance specialist ensuring policies align with regulatory frameworks "
        "and identifying any compliance gaps."
    ),
    verbose=True,
    # model= 'gpt-4-turbo'
    model=call_llama 
)

policy_impact_assessment_agent = Agent(
    role="Policy Impact Assessment Manager",
    goal="Assess policy compliance impacts and provide actionable recommendations.",
    backstory=(
        "An analytical expert responsible for automating compliance reports, "
        "categorizing impact levels, and updating policies based on outcomes."
    ),
    verbose=True,
    # model= 'gpt-4-turbo'
    model=call_llama 
)

compliance_reviewer_agent = Agent(
    role="Compliance Reviewer",
    goal="Ensure policies adhere to regulations and provide gap reports.",
    backstory=(
        "A meticulous reviewer who verifies adherence to standards, "
        "identifies compliance gaps, and ensures policy updates are thorough."
    ),
    verbose=True,
    # model= 'gpt-4-turbo'
    model=call_llama 
)

### Defining the Tasks

In [6]:
# Defining Tasks

draft_policy_task = Task(
    description=(
        "Draft a detailed cybersecurity policy using the latest Cyber Security standards for {organization}. Ensure alignment "
        "with the organization's objectives and regulatory requirements."
        "Write a document with at least 3 pages and detail it in Sections and subtitles."
    ),
    expected_output="An initial comprehensive cybersecurity policy draft in markdown format.",
    agent=policy_writer_agent
)

review_policy_task = Task(
    description=(
        "Review the drafted cybersecurity policy for adherence to standards, grammar, and overall structure. "
        "Provide structured feedback for improvement."
    ),
    expected_output="A revised comprehensive Cyber Security policy draft with feedback and suggestions.",
    agent=policy_reviewer_agent
)

map_compliance_task = Task(
    description=(
        "Analyze the policy for its alignment with compliance standards. Identify gaps, map compliance needs, "
        "and create an initial action plan."
    ),
    expected_output="A Cyber Security Policy and a compliance mapping report with action plans for identified gaps.",
    agent=compliance_policy_mapper_agent
)

impact_assessment_task = Task(
    description=(
        "Automate compliance reports, categorize impact levels, and handle recommendations "
        "for policy adjustments based on compliance outcomes."
    ),
    expected_output="A comprehensive Cyber Security Policy and a detailed compliance impact report with categorized recommendations for {organization}.",
    agent=policy_impact_assessment_agent
)

final_compliance_review_task = Task(
    description=(
        "Perform a final review of the policy to ensure it adheres to regulations and standards. "
        "Produce a compliance gap report with suggestions for improvement and final updates."
        "Keep both the revised comprehensive Cyber Security Policy and Compliance GAP Analysis in the final report for {organization}."
    ),
    expected_output="A final compliance gap report and updated policy.",
    agent=compliance_reviewer_agent
)


### Defining the Crew

In [8]:
# Defining the Crew
crew = Crew(
    agents=[
        policy_writer_agent,
        policy_reviewer_agent,
        compliance_policy_mapper_agent,
        policy_impact_assessment_agent,
        compliance_reviewer_agent
    ],
    tasks=[
        draft_policy_task,
        review_policy_task,
        map_compliance_task,
        impact_assessment_task,
        final_compliance_review_task
    ],
    verbose=2
)

2024-12-13 00:26:14,428 - 8662727360 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


### Running the Crew

In [10]:
# Running the Crew
result = crew.kickoff(inputs={"organization": "HAVI"})


 [DEBUG]: == Working Agent: Policy Writer
 [INFO]: == Starting Task: Draft a detailed cybersecurity policy using the latest Cyber Security standards for HAVI. Ensure alignment with the organization's objectives and regulatory requirements.Write a document with at least 3 pages and detail it in Sections and subtitles.


> Entering new CrewAgentExecutor chain...
The first step in drafting a cybersecurity policy is understanding the organization's objectives, regulatory requirements, and current cybersecurity standards. I need to consider the type of data we're dealing with, the possible threats, and the measures to protect the data. Also, I need to take into account the company's culture, its approach to balancing security with business needs. To get this information, I will delegate tasks and ask questions to my co-workers.

Action: 
Delegate work to co-worker

Action Input: 
{
  "coworker": "Compliance-Policy Mapper", 
  "task": "Identify the regulatory requirements for HAVI", 
  "cont

### Saving the Generated Documents in Markdown format

In [11]:
# Saving the Result
output_file = "CyberSecurityPolicy_DEMO.md"
with open(output_file, "w") as file:
    file.write(result)

print(f"Policy document saved to {output_file}.")

Policy document saved to CyberSecurityPolicy_DEMO.md.
